# Turing Test

## Import packages

In [ ]:
import os
import re
import sys
import math
import random
import numpy as np
import pandas as pd
import krippendorff
import statsmodels.api as sm
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import clear_output
from scipy.stats import chi2_contingency
from statsmodels.stats.inter_rater import aggregate_raters

DEEPSEEK_BASE_URL = "https://api.deepseek.com"

current_dir = os.path.dirname(os.path.abspath("__file__"))
main_dir = os.path.join(current_dir, '..')
sys.path.append(main_dir)

from Humanise.humanise import humanise_sentence, initialise_globals

initialise_globals(main_dir)
HUMAN_DATAPATH = os.path.join(current_dir, 'human.txt')
GENERATE_DATAPATH = os.path.join(current_dir, 'Turing2/synthetic_generate_v2.txt')
HUMANISE_DATAPATH = os.path.join(current_dir, 'Turing2/synthetic_humanise_v2.txt')

# Load environment variables
load_dotenv()
api_key = os.getenv("API_KEY")
client = OpenAI(api_key=api_key, base_url=DEEPSEEK_BASE_URL)

## Prepare Turing Test data

In [5]:
# Function to load human data from MaintNorm dataset
def load_maintnorm_sentences(file_path):
    dirty_sentences = []
    clean_sentences = []
    current_dirty = []
    current_clean = []
    pattern = r'[A-Za-z]{2}\d{4}-'
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:  # Empty line means a new sentence
                if current_dirty and current_clean:
                    dirty_sentences.append(' '.join(current_dirty))
                    clean_sentences.append(' '.join(current_clean))
                    current_dirty = []
                    current_clean = []
            else:
                parts = line.split('\t')
                if len(parts) > 1:
                    dirty, clean = parts[0], parts[1]
                    if not clean in ['<id>', '-']:
                        dirty = re.sub(pattern, '', dirty)
                        current_dirty.append(dirty.lower().strip())
                    current_clean.append(clean.lower())
    return dirty_sentences, clean_sentences

# Save all human data to text file
def save_human_data():
    train_dirty, _ = load_maintnorm_sentences('../data/MaintNorm/train.norm')
    test_dirty, _ = load_maintnorm_sentences('../data/MaintNorm/test.norm')
    val_dirty, _ = load_maintnorm_sentences('../data/MaintNorm/val.norm')
    full_dirty = train_dirty + test_dirty + val_dirty
    human_data = list(set(full_dirty)) # remove duplicates
    with open(HUMAN_DATAPATH, 'w') as f:
        for item in human_data:
            if any(char.isdigit() for char in item):
                # Reduce probability for strings containing numbers
                if random.random() > 0.25:  # 80% chance to skip
                    continue
            f.write("%s\n" % item)

# Function to load human sentences or synthetic sentences
def load_sentences(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file]

# Function to sample by sections
def synthetic_sample(data, num_samples, num_sections):
    section_size = len(data) // num_sections
    syn_sections = [data[i*section_size:(i+1)*section_size] for i in range(num_sections)]
    i = 0
    for d in data[num_sections*section_size:]:
        syn_sections[i].append(d)
        i += 1
    sample_size = math.ceil(num_samples / num_sections)
    syn_samples = []
    for section in syn_sections:
        syn_samples.extend(random.sample(section, sample_size))
    syn_samples = random.sample(syn_samples, 50)
    return syn_samples

# Function to generate random labels to test performance
def random_labels(turing):
    turing = pd.read_csv(turing)
    turing['label'] = np.random.choice(['h', 's'], turing.shape[0])
    turing.to_csv('Turing2/turing_random.csv', index=False)

# Uncomment to save human data to text file
save_human_data()

# Humanise generated synthetic data
# synthetic_data = load_sentences(GENERATE_DATAPATH)
# humanise_data = [humanise_sentence(s) for s in synthetic_data]

# with open(HUMANISE_DATAPATH, 'w') as f:
#     for item in humanise_data:
#         f.write("%s\n" % item)

In [15]:
# Random 50 human data sentences
human_data = load_sentences(HUMAN_DATAPATH)
human_50 = random.sample(human_data, 50)
human_50 = pd.DataFrame(human_50, columns=['sentence'])
human_50['label'] = 'h'

# Random 50 synthetic data sentences
synthetic_data = load_sentences(HUMANISE_DATAPATH)
synthetic_50 = synthetic_sample(synthetic_data, 50, 8)
synthetic_50 = pd.DataFrame(synthetic_50, columns=['sentence'])
synthetic_50['label'] = 's'

# Combine and shuffle human and synthetic data
turing_data = pd.concat([human_50, synthetic_50])
turing_data = turing_data.sample(frac=1).reset_index(drop=True)

turing_data.to_csv('target.csv', index=False)
turing_data.drop(columns=['label']).to_csv('turing.csv', index=False)

## Evaluate annotators on Turing Test data

### Individual evaluation functions

In [4]:
def evaluate_turing(target_file, turing_file, print_results=True):
    # Read files
    target = pd.read_csv(target_file)
    turing = pd.read_csv(turing_file)
    turing['label'] = turing['label'].str.lower()
    combine = pd.merge(target, turing, on='sentence')

    # Counts
    tp = ((combine['label_x'] == 'h') & (combine['label_y'] == 'h')).sum()
    tn = ((combine['label_x'] == 's') & (combine['label_y'] == 's')).sum()
    fp = ((combine['label_x'] == 's') & (combine['label_y'] == 'h')).sum()
    fn = ((combine['label_x'] == 'h') & (combine['label_y'] == 's')).sum()

    # Chi-square test
    contingency_table = [[tp, fp], [fn, tn]]
    res = chi2_contingency(contingency_table)

    # Confusion matrix
    column_names = ['Actual Human', 'Actual Synthetic']
    index_names = ['Predicted Human', 'Predicted Synthetic']
    confusion_matrix = pd.DataFrame(contingency_table, columns=column_names, index=index_names)

    # Accuracy, Precision, Recall, F1-score
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1_score = 2 * (precision * recall) / (precision + recall)
    
    # Percentage of human and synthetic data
    num_label_h = turing['label'].value_counts()['h']
    num_label_s = turing['label'].value_counts()['s']
    label_h_percentage = num_label_h / len(turing)
    label_s_percentage = num_label_s / len(turing)
    
    # Probability of synthetic data being labelled as human
    num_target_s = target['label'].value_counts()['s']
    prob_s_labelled_h = fp / num_target_s

    # Print results
    if print_results:
        print('-------------------------------- Frequency of Labels')
        print(f'Labelled human      : {num_label_h} ({label_h_percentage:.2f})')
        print(f'Labelled synthetic  : {num_label_s} ({label_s_percentage:.2f})')
        
        print('---------------------------------- Confusion Matrix')
        print(confusion_matrix)
        
        print('----------------------------------- Chi-Square Test')
        print(f'Chi-square          : {res.statistic:.4f}')
        print(f'p-value             : {res.pvalue:.4f}')
        print(f'Degrees of freedom  : {res.dof}')
        print('Expected frequencies:')
        print(res.expected_freq)

        print('--------------------------------------- Performance')
        print(f'Accuracy            : {accuracy:.3f}')
        print(f'Precision           : {precision:.3f}')
        print(f'Recall              : {recall:.3f}')
        print(f'F1-score            : {f1_score:.3f}')
        print(f'Probability S as H  : {prob_s_labelled_h:.3f}')

    return (accuracy, precision, recall, f1_score, label_h_percentage, prob_s_labelled_h)

def filter_predictions(target_file, turing_file):
    target = pd.read_csv(target_file)
    turing = pd.read_csv(turing_file) 
    turing['label'] = turing['label'].str.lower()
    combine = pd.merge(target, turing, on='sentence')

    # Actually human, predicted synthetic
    false_synthetic = combine[(combine['label_x'] == 'h') & (combine['label_y'] == 's')]['sentence']
    # Actually synthetic, predicted human
    false_human = combine[(combine['label_x'] == 's') & (combine['label_y'] == 'h')]['sentence']
    
    print('----------------------------------- False Synthetic')
    for s in false_synthetic:
        print(s)
    print('--------------------------------------- False Human')
    for s in false_human:
        print(s)

### Agreement evaluation functions

In [2]:
def average_performance(turing_files, accuracies, precisions, recalls, f1_scores, human_percentages, probs):
    # Calculate overall average performance
    avg_accuracy = sum(accuracies) / len(accuracies)
    avg_precision = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    avg_f1_score = sum(f1_scores) / len(f1_scores)
    avg_human_percentage = sum(human_percentages) / len(human_percentages)
    avg_probability = sum(probs) / len(probs)
    alpha_agreement = annotator_agreement(turing_files)
    kappa_agreement = fleiss_kappa(turing_files)
    
    # Print results
    print('----------------------------------- Overall Results')
    print(f'Average Accuracy    : {avg_accuracy:.4f}')
    print(f'Average Precision   : {avg_precision:.4f}')
    print(f'Average Recall      : {avg_recall:.4f}')
    print(f'Average F1-score    : {avg_f1_score:.4f}')
    print(f'Average Human %     : {avg_human_percentage:.4f}')
    print(f'Krippendorff Alpha  : {alpha_agreement:.4f}')
    print(f'Fleiss Kappa        : {kappa_agreement:.4f}')
    print(f'Average Probability : {avg_probability:.4f}')
    return (avg_accuracy, avg_precision, avg_recall, avg_f1_score, avg_human_percentage, alpha_agreement, avg_probability)

def find_common(target_file, turing_files, threshold=0.8):
    # Regex pattern extract name from filename
    pattern = r'(?<=_)[^_]+(?=_v\d+\.csv)'
    df = pd.read_csv(target_file)
    for file in turing_files:
        name = re.search(pattern, file).group()
        pred = pd.read_csv(file)['label']
        df[name] = pred
        
    num_evaluators = len(turing_files)
    min_required = int(num_evaluators * threshold)
    
    # Find when everyone predicted human and actually synthetic
    common_fp = df[(df['label'] == 's') & (df.iloc[:, 2:].eq('h').sum(axis=1) >= min_required)]
    # Find when everyone predicted synthetic and actually human
    common_fn = df[(df['label'] == 'h') & (df.iloc[:, 2:].eq('s').sum(axis=1) >= min_required)]
    # Find when everyone predicted human and actually human
    common_tp = df[(df['label'] == 'h') & (df.iloc[:, 2:].eq('h').sum(axis=1) >= min_required)]
    # Find when everyone predicted synthetic and actually synthetic
    common_tn = df[(df['label'] == 's') & (df.iloc[:, 2:].eq('s').sum(axis=1) >= min_required)]

    def print_common_results(label, common_df, condition_label):
        print(f'--------------------- Common {condition_label} ({len(common_df)})')
        for idx, row in common_df.iterrows():
            sentence = row.iloc[0]
            count = row.iloc[2:].eq(label).sum()  # Count how many evaluators agreed
            print(f'{sentence:40} ({count}/{num_evaluators})')
    
    print_common_results('h', common_fp, 'Predicted Human Actually Synthetic')
    print_common_results('s', common_fn, 'Predicted Synthetic Actually Human')
    print_common_results('s', common_tn, 'Predicted Synthetic Actually Synthetic')
    print_common_results('h', common_tp, 'Predicted Human Actually Human')

    total_common = len(common_fp) + len(common_fn) + len(common_tp) + len(common_tn)
    print('\nTotal commonly labelled:', total_common)

# Krippendorff's Alpha for annotator agreement
def annotator_agreement(turing_files):
    labels = [pd.read_csv(file)['label'].tolist() for file in turing_files]
    alpha = krippendorff.alpha(reliability_data=labels, level_of_measurement='nominal')
    return alpha

# Fleiss' Kappa for annotator agreement
def fleiss_kappa(turing_files):
    labels = [pd.read_csv(file)['label'].tolist() for file in turing_files]
    labels = np.array(pd.DataFrame(labels).T)
    category_map = {'h': 1, 's': 0}
    numeric_data = np.array([[category_map[label] for label in item] for item in labels])
    contingency_table, _ = aggregate_raters(numeric_data, n_cat=2)
    kappa = sm.stats.fleiss_kappa(contingency_table)
    return kappa

# Combine all annotator results into one dataframe
def combine_results(target_file, turing_files, outfile):
    target_df = pd.read_csv(target_file)
    combined = []
    confusion_matrices = []
    for annotator_idx, file in enumerate(turing_files):
        annotator_df = pd.read_csv(file)
        merged_df = pd.merge(target_df, annotator_df, on='sentence', suffixes=('_target', '_guess'))
        
        # Populate combined results dataframe
        for sentence_idx, row in merged_df.iterrows():
            combined.append({
                'Sentence': sentence_idx+1,
                'Target': row['label_target'],
                'Guess': row['label_guess'],
                'Annotator': annotator_idx+1
            })
        
        # Populate confusion matrix
        tp = ((merged_df['label_target'] == 'h') & (merged_df['label_guess'] == 'h')).sum()
        tn = ((merged_df['label_target'] == 's') & (merged_df['label_guess'] == 's')).sum()
        fp = ((merged_df['label_target'] == 's') & (merged_df['label_guess'] == 'h')).sum()
        fn = ((merged_df['label_target'] == 'h') & (merged_df['label_guess'] == 's')).sum()
        contingency_table = [[tp, fp], [fn, tn]]
        confusion_matrix = pd.DataFrame(
            contingency_table, 
            columns=['Actual Human', 'Actual Synthetic'], 
            index=['Predicted Human', 'Predicted Synthetic']
        )
        confusion_matrix['Annotator'] = annotator_idx + 1
        confusion_matrices.append(confusion_matrix.reset_index())

    # Save combined results
    combined_df = pd.DataFrame(combined)
    combined_df.sort_values(by=['Sentence', 'Annotator'], inplace=True)
    combined_df.to_csv(outfile.replace('.csv', '_combined.csv'), index=False)
    
    # Save confusion matrices
    confusion_matrices_df = pd.concat(confusion_matrices)
    confusion_matrices_df.rename(columns={'index': 'Prediction'}, inplace=True)
    confusion_matrices_df.to_csv(outfile.replace('.csv', '_confusion_matrices.csv'), index=False)

# Turing Test 1

## Evaluate individual predictions V1

In [4]:
_ = evaluate_turing('Turing1/target_v1.csv', 'Turing1/turing_jf_v1.csv')
# filter_predictions('Turing1/target_v1.csv', 'Turing1/turing_jf_v1.csv')

-------------------------------- Frequency of Labels
Labelled human      : 54 (0.54)
Labelled synthetic  : 45 (0.45)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                29                25
Predicted Synthetic            20                25
----------------------------------- Chi-Square Test
Chi-square          : 0.5122
p-value             : 0.4742
Degrees of freedom  : 1
Expected frequencies:
[[26.72727273 27.27272727]
 [22.27272727 22.72727273]]
--------------------------------------- Performance
Accuracy            : 0.545
Precision           : 0.537
Recall              : 0.592
F1-score            : 0.563
Probability S as H  : 0.500


In [5]:
_ = evaluate_turing('Turing1/target_v1.csv', 'Turing1/turing_cg_v1.csv')
# filter_predictions('Turing1/target_v1.csv', 'Turing1/turing_cg_v1.csv')

-------------------------------- Frequency of Labels
Labelled human      : 56 (0.56)
Labelled synthetic  : 44 (0.44)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                47                 9
Predicted Synthetic             3                41
----------------------------------- Chi-Square Test
Chi-square          : 55.5601
p-value             : 0.0000
Degrees of freedom  : 1
Expected frequencies:
[[28. 28.]
 [22. 22.]]
--------------------------------------- Performance
Accuracy            : 0.880
Precision           : 0.839
Recall              : 0.940
F1-score            : 0.887
Probability S as H  : 0.180


In [6]:
_ = evaluate_turing('Turing1/target_v1.csv', 'Turing1/turing_ms_v1.csv')
# filter_predictions('Turing1/target_v1.csv', 'Turing1/turing_ms_v1.csv')

-------------------------------- Frequency of Labels
Labelled human      : 65 (0.65)
Labelled synthetic  : 35 (0.35)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                40                25
Predicted Synthetic            10                25
----------------------------------- Chi-Square Test
Chi-square          : 8.6154
p-value             : 0.0033
Degrees of freedom  : 1
Expected frequencies:
[[32.5 32.5]
 [17.5 17.5]]
--------------------------------------- Performance
Accuracy            : 0.650
Precision           : 0.615
Recall              : 0.800
F1-score            : 0.696
Probability S as H  : 0.500


In [7]:
_ = evaluate_turing('Turing1/target_v1.csv', 'Turing1/turing_mh_v1.csv')
# filter_predictions('Turing1/target_v1.csv', 'Turing1/turing_mh_v1.csv')

-------------------------------- Frequency of Labels
Labelled human      : 40 (0.40)
Labelled synthetic  : 60 (0.60)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                32                 8
Predicted Synthetic            18                42
----------------------------------- Chi-Square Test
Chi-square          : 22.0417
p-value             : 0.0000
Degrees of freedom  : 1
Expected frequencies:
[[20. 20.]
 [30. 30.]]
--------------------------------------- Performance
Accuracy            : 0.740
Precision           : 0.800
Recall              : 0.640
F1-score            : 0.711
Probability S as H  : 0.160


In [8]:
_ = evaluate_turing('Turing1/target_v1.csv', 'Turing1/turing_cw_v1.csv')
# filter_predictions('Turing1/target_v1.csv', 'Turing1/turing_cw_v1.csv')

-------------------------------- Frequency of Labels
Labelled human      : 52 (0.52)
Labelled synthetic  : 48 (0.48)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                27                25
Predicted Synthetic            23                25
----------------------------------- Chi-Square Test
Chi-square          : 0.0401
p-value             : 0.8414
Degrees of freedom  : 1
Expected frequencies:
[[26. 26.]
 [24. 24.]]
--------------------------------------- Performance
Accuracy            : 0.520
Precision           : 0.519
Recall              : 0.540
F1-score            : 0.529
Probability S as H  : 0.500


## Evaluate annotator agreement V1

In [10]:
evaluators_v1 = [
    'Turing1/turing_ms_v1.csv',
    'Turing1/turing_mh_v1.csv',
    'Turing1/turing_cw_v1.csv'
]

accuracy, precision, recall, f1_score, human_percentage, prob_s_labelled_h = [], [], [], [], [], []
for e in evaluators_v1:
    results = evaluate_turing('Turing1/target_v1.csv', e, False)
    accuracy.append(results[0])
    precision.append(results[1])
    recall.append(results[2])
    f1_score.append(results[3])
    human_percentage.append(results[4])
    prob_s_labelled_h.append(results[5])

overall_results = average_performance(evaluators_v1, accuracy, precision, recall, 
                                      f1_score, human_percentage, prob_s_labelled_h)

----------------------------------- Overall Results
Average Accuracy    : 0.6367
Average Precision   : 0.6449
Average Recall      : 0.6600
Average F1-score    : 0.6454
Average Human %     : 0.5233
Krippendorff Alpha  : 0.1743
Fleiss Kappa        : 0.1715
Average Probability : 0.3867


In [13]:
find_common('Turing1/target_v1.csv', evaluators_v1, 1.0)

--------------------- Common Predicted Human Actually Synthetic (5)
replace leaking lube pump                (3/3)
hmu leaking hydraulic fluid              (3/3)
decking has several cracks               (3/3)
boom foot clevbis pin has no grease      (3/3)
diff lube hose insp for leaks            (3/3)
--------------------- Common Predicted Synthetic Actually Human (4)
replace pos 8 wheel end po               (3/3)
replace faulty brake sensor1 task        (3/3)
cw coolant leak from #15 cylind          (3/3)
pcr room over alarm                      (3/3)
--------------------- Common Predicted Synthetic Actually Synthetic (13)
chge out leaking axle oil cool           (3/3)
leak detected infan pump                 (3/3)
cabindoor is leaking                     (3/3)
leak in air aircon hose                  (3/3)
replace leaking air aircon hose          (3/3)
plug has a leak                          (3/3)
leaikng fluid fr swingbrake pump         (3/3)
gasket isleaking                       

# Turing Test 2

## Evaluate individual predictions V2

In [14]:
_ = evaluate_turing('Turing2/target_v2.csv', 'Turing2/turing_cg_v2.csv')
# filter_predictions('Turing2/target_v2.csv', 'Turing2/turing_cg_v2.csv')

-------------------------------- Frequency of Labels
Labelled human      : 48 (0.48)
Labelled synthetic  : 52 (0.52)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                31                17
Predicted Synthetic            19                33
----------------------------------- Chi-Square Test
Chi-square          : 6.7708
p-value             : 0.0093
Degrees of freedom  : 1
Expected frequencies:
[[24. 24.]
 [26. 26.]]
--------------------------------------- Performance
Accuracy            : 0.640
Precision           : 0.646
Recall              : 0.620
F1-score            : 0.633
Probability S as H  : 0.340


In [15]:
_ = evaluate_turing('Turing2/target_v2.csv', 'Turing2/turing_jf_v2.csv')
# filter_predictions('Turing2/target_v2.csv', 'Turing2/turing_jf_v2.csv')

-------------------------------- Frequency of Labels
Labelled human      : 49 (0.49)
Labelled synthetic  : 51 (0.51)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                25                24
Predicted Synthetic            25                26
----------------------------------- Chi-Square Test
Chi-square          : 0.0000
p-value             : 1.0000
Degrees of freedom  : 1
Expected frequencies:
[[24.5 24.5]
 [25.5 25.5]]
--------------------------------------- Performance
Accuracy            : 0.510
Precision           : 0.510
Recall              : 0.500
F1-score            : 0.505
Probability S as H  : 0.480


In [16]:
_ = evaluate_turing('Turing2/target_v2.csv', 'Turing2/turing_ms_v2.csv')
# filter_predictions('Turing2/target_v2.csv', 'Turing2/turing_ms_v2.csv')

-------------------------------- Frequency of Labels
Labelled human      : 63 (0.63)
Labelled synthetic  : 37 (0.37)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                37                26
Predicted Synthetic            13                24
----------------------------------- Chi-Square Test
Chi-square          : 4.2900
p-value             : 0.0383
Degrees of freedom  : 1
Expected frequencies:
[[31.5 31.5]
 [18.5 18.5]]
--------------------------------------- Performance
Accuracy            : 0.610
Precision           : 0.587
Recall              : 0.740
F1-score            : 0.655
Probability S as H  : 0.520


In [17]:
_ = evaluate_turing('Turing2/target_v2.csv', 'Turing2/turing_mh_v2.csv')
# filter_predictions('Turing2/target_v2.csv', 'Turing2/turing_mh_v2.csv')

-------------------------------- Frequency of Labels
Labelled human      : 64 (0.64)
Labelled synthetic  : 36 (0.36)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                36                28
Predicted Synthetic            14                22
----------------------------------- Chi-Square Test
Chi-square          : 2.1267
p-value             : 0.1447
Degrees of freedom  : 1
Expected frequencies:
[[32. 32.]
 [18. 18.]]
--------------------------------------- Performance
Accuracy            : 0.580
Precision           : 0.562
Recall              : 0.720
F1-score            : 0.632
Probability S as H  : 0.560


In [18]:
_ = evaluate_turing('Turing2/target_v2.csv', 'Turing2/turing_cw_v2.csv')
# filter_predictions('Turing2/target_v2.csv', 'Turing2/turing_cw_v2.csv')

-------------------------------- Frequency of Labels
Labelled human      : 53 (0.53)
Labelled synthetic  : 47 (0.47)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                28                25
Predicted Synthetic            22                25
----------------------------------- Chi-Square Test
Chi-square          : 0.1606
p-value             : 0.6886
Degrees of freedom  : 1
Expected frequencies:
[[26.5 26.5]
 [23.5 23.5]]
--------------------------------------- Performance
Accuracy            : 0.530
Precision           : 0.528
Recall              : 0.560
F1-score            : 0.544
Probability S as H  : 0.500


## Evaluate annotator agreement V2

In [21]:
evaluators_v2 = [
    'Turing2/turing_ms_v2.csv',
    'Turing2/turing_mh_v2.csv',
    'Turing2/turing_cw_v2.csv'
]

accuracy, precision, recall, f1_score, human_percentage, prob_s_labelled_h = [], [], [], [], [], []
for e in evaluators_v2:
    results = evaluate_turing('Turing2/target_v2.csv', e, False)
    accuracy.append(results[0])
    precision.append(results[1])
    recall.append(results[2])
    f1_score.append(results[3])
    human_percentage.append(results[4])
    prob_s_labelled_h.append(results[5])
    
overall_results = average_performance(evaluators_v2, accuracy, precision, recall, 
                                      f1_score, human_percentage, prob_s_labelled_h)

----------------------------------- Overall Results
Average Accuracy    : 0.5733
Average Precision   : 0.5594
Average Recall      : 0.6733
Average F1-score    : 0.6100
Average Human %     : 0.6000
Krippendorff Alpha  : 0.1002
Fleiss Kappa        : 0.0972
Average Probability : 0.5267


In [22]:
find_common('Turing2/target_v2.csv', evaluators_v2, 1.0)

--------------------- Common Predicted Human Actually Synthetic (10)
change out w/pump sft unserviceable      (3/3)
coolant pipe clamps require loctite      (3/3)
d/line air conditioner not producing cold (3/3)
loctite needed on brk pump bolt          (3/3)
repair crack in boom chofd               (3/3)
apply loctite on clamps                  (3/3)
rep crack in rock deflector              (3/3)
auto-lube system fault                   (3/3)
leak in condenser fan hose               (3/3)
no grease in dogbone pin                 (3/3)
--------------------- Common Predicted Synthetic Actually Human (2)
fabricate radiator stands                (3/3)
clar hr air con servic                   (3/3)
--------------------- Common Predicted Synthetic Actually Synthetic (7)
hand is cracked                          (3/3)
leak found in fan pump oil               (3/3)
swing brake hose is in need of rep       (3/3)
parts washer pump chip sensor alarm      (3/3)
boarding ladder is not wor king       

# Turing Test 3

## Evaluate individual predictions V3

In [23]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_mh_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                33                17
Predicted Synthetic            17                33
----------------------------------- Chi-Square Test
Chi-square          : 9.0000
p-value             : 0.0027
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.660
Precision           : 0.660
Recall              : 0.660
F1-score            : 0.660
Probability S as H  : 0.340


In [24]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_cw_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                26                24
Predicted Synthetic            24                26
----------------------------------- Chi-Square Test
Chi-square          : 0.0400
p-value             : 0.8415
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.520
Precision           : 0.520
Recall              : 0.520
F1-score            : 0.520
Probability S as H  : 0.480


In [25]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_jn_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                18                32
Predicted Synthetic            32                18
----------------------------------- Chi-Square Test
Chi-square          : 6.7600
p-value             : 0.0093
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.360
Precision           : 0.360
Recall              : 0.360
F1-score            : 0.360
Probability S as H  : 0.640


In [26]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_js_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                20                30
Predicted Synthetic            30                20
----------------------------------- Chi-Square Test
Chi-square          : 3.2400
p-value             : 0.0719
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.400
Precision           : 0.400
Recall              : 0.400
F1-score            : 0.400
Probability S as H  : 0.600


In [27]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_mm_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                29                21
Predicted Synthetic            21                29
----------------------------------- Chi-Square Test
Chi-square          : 1.9600
p-value             : 0.1615
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.580
Precision           : 0.580
Recall              : 0.580
F1-score            : 0.580
Probability S as H  : 0.420


In [28]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_am_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                25                25
Predicted Synthetic            25                25
----------------------------------- Chi-Square Test
Chi-square          : 0.0000
p-value             : 1.0000
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.500
Precision           : 0.500
Recall              : 0.500
F1-score            : 0.500
Probability S as H  : 0.500


In [29]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_tg_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                32                18
Predicted Synthetic            18                32
----------------------------------- Chi-Square Test
Chi-square          : 6.7600
p-value             : 0.0093
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.640
Precision           : 0.640
Recall              : 0.640
F1-score            : 0.640
Probability S as H  : 0.360


In [30]:
_ = evaluate_turing('Turing3/target_v3.csv', 'Turing3/turing_sy_v3.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                24                26
Predicted Synthetic            26                24
----------------------------------- Chi-Square Test
Chi-square          : 0.0400
p-value             : 0.8415
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.480
Precision           : 0.480
Recall              : 0.480
F1-score            : 0.480
Probability S as H  : 0.520


## Evaluate annotator agreement V3

In [6]:
evaluators_v3 = [
    'Turing3/turing_cw_v3.csv',
    'Turing3/turing_am_v3.csv',
    'Turing3/turing_jn_v3.csv',
    'Turing3/turing_js_v3.csv',
    'Turing3/turing_mm_v3.csv',
    'Turing3/turing_tg_v3.csv',
    'Turing3/turing_sy_v3.csv'
]

accuracy, precision, recall, f1_score, human_percentage, prob_s_labelled_h = [], [], [], [], [], []
for e in evaluators_v3:
    results = evaluate_turing('Turing3/target_v3.csv', e, False)
    accuracy.append(results[0])
    precision.append(results[1])
    recall.append(results[2])
    f1_score.append(results[3])
    human_percentage.append(results[4])
    prob_s_labelled_h.append(results[5])

overall_results = average_performance(evaluators_v3, accuracy, precision, recall, 
                                      f1_score, human_percentage, prob_s_labelled_h)

combine_results('Turing3/target_v3.csv', evaluators_v3, 'Turing3/v3.csv')

----------------------------------- Overall Results
Average Accuracy    : 0.4971
Average Precision   : 0.4971
Average Recall      : 0.4971
Average F1-score    : 0.4971
Average Human %     : 0.5000
Krippendorff Alpha  : -0.0005
Fleiss Kappa        : -0.0019
Average Probability : 0.5029


In [32]:
find_common('Turing3/target_v3.csv', evaluators_v3, 0.9)

--------------------- Common Predicted Human Actually Synthetic (6)
dropped cell in battery                  (6/7)
repair crack in rock deflector           (6/7)
h/rail is cracked                        (6/7)
cooler pump bolt needsloctite            (6/7)
diff oil is leaking                      (6/7)
cab needs an aerial                      (7/7)
--------------------- Common Predicted Synthetic Actually Human (4)
replace injectors and o                  (6/7)
42347 monthly ansul system service       (6/7)
drag tun roller bracket/bearings         (6/7)
repair bucket in surf w/shop             (6/7)
--------------------- Common Predicted Synthetic Actually Synthetic (3)
revolving frame has oil                  (6/7)
replace the bolt                         (6/7)
leak of hyd oil from cooler fan hose     (6/7)
--------------------- Common Predicted Human Actually Human (1)
replace uni's on rear driveline          (6/7)

Total commonly labelled: 14


# Turing Test 4

## Evaluate individual predictions V4

In [33]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_mh_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                32                18
Predicted Synthetic            18                32
----------------------------------- Chi-Square Test
Chi-square          : 6.7600
p-value             : 0.0093
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.640
Precision           : 0.640
Recall              : 0.640
F1-score            : 0.640
Probability S as H  : 0.360


In [34]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_cw_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                30                20
Predicted Synthetic            20                30
----------------------------------- Chi-Square Test
Chi-square          : 3.2400
p-value             : 0.0719
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.600
Precision           : 0.600
Recall              : 0.600
F1-score            : 0.600
Probability S as H  : 0.400


In [35]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_am_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                21                29
Predicted Synthetic            29                21
----------------------------------- Chi-Square Test
Chi-square          : 1.9600
p-value             : 0.1615
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.420
Precision           : 0.420
Recall              : 0.420
F1-score            : 0.420
Probability S as H  : 0.580


In [36]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_am2_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                24                26
Predicted Synthetic            26                24
----------------------------------- Chi-Square Test
Chi-square          : 0.0400
p-value             : 0.8415
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.480
Precision           : 0.480
Recall              : 0.480
F1-score            : 0.480
Probability S as H  : 0.520


In [37]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_jn_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                29                21
Predicted Synthetic            21                29
----------------------------------- Chi-Square Test
Chi-square          : 1.9600
p-value             : 0.1615
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.580
Precision           : 0.580
Recall              : 0.580
F1-score            : 0.580
Probability S as H  : 0.420


In [38]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_js_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                19                31
Predicted Synthetic            31                19
----------------------------------- Chi-Square Test
Chi-square          : 4.8400
p-value             : 0.0278
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.380
Precision           : 0.380
Recall              : 0.380
F1-score            : 0.380
Probability S as H  : 0.620


In [39]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_mm_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                22                28
Predicted Synthetic            28                22
----------------------------------- Chi-Square Test
Chi-square          : 1.0000
p-value             : 0.3173
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.440
Precision           : 0.440
Recall              : 0.440
F1-score            : 0.440
Probability S as H  : 0.560


In [40]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_tg_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                32                18
Predicted Synthetic            18                32
----------------------------------- Chi-Square Test
Chi-square          : 6.7600
p-value             : 0.0093
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.640
Precision           : 0.640
Recall              : 0.640
F1-score            : 0.640
Probability S as H  : 0.360


In [41]:
_ = evaluate_turing('Turing4/target_v4.csv', 'Turing4/turing_sy_v4.csv')

-------------------------------- Frequency of Labels
Labelled human      : 50 (0.50)
Labelled synthetic  : 50 (0.50)
---------------------------------- Confusion Matrix
                     Actual Human  Actual Synthetic
Predicted Human                29                21
Predicted Synthetic            21                29
----------------------------------- Chi-Square Test
Chi-square          : 1.9600
p-value             : 0.1615
Degrees of freedom  : 1
Expected frequencies:
[[25. 25.]
 [25. 25.]]
--------------------------------------- Performance
Accuracy            : 0.580
Precision           : 0.580
Recall              : 0.580
F1-score            : 0.580
Probability S as H  : 0.420


## Evaluate annotator agreement V4

In [7]:
evaluators_v4 = [
    'Turing4/turing_cw_v4.csv',
    'Turing4/turing_am_v4.csv',
    'Turing4/turing_jn_v4.csv',
    'Turing4/turing_js_v4.csv',
    'Turing4/turing_mm_v4.csv',
    'Turing4/turing_tg_v4.csv',
    'Turing4/turing_sy_v4.csv',
]

accuracy, precision, recall, f1_score, human_percentage, prob_s_labelled_h = [], [], [], [], [], []
for e in evaluators_v4:
    results = evaluate_turing('Turing4/target_v4.csv', e, False)
    accuracy.append(results[0])
    precision.append(results[1])
    recall.append(results[2])
    f1_score.append(results[3])
    human_percentage.append(results[4])
    prob_s_labelled_h.append(results[5])

overall_results = average_performance(evaluators_v4, accuracy, precision, recall, 
                                      f1_score, human_percentage, prob_s_labelled_h)

combine_results('Turing4/target_v4.csv', evaluators_v4, 'Turing4/v4.csv')

----------------------------------- Overall Results
Average Accuracy    : 0.5200
Average Precision   : 0.5200
Average Recall      : 0.5200
Average F1-score    : 0.5200
Average Human %     : 0.5000
Krippendorff Alpha  : -0.0024
Fleiss Kappa        : -0.0038
Average Probability : 0.4800


In [43]:
find_common('Turing4/target_v4.csv', evaluators_v4, 0.9)

--------------------- Common Predicted Human Actually Synthetic (2)
diff oil is leaking                      (6/7)
install missing smoke detector in cab    (6/7)
--------------------- Common Predicted Synthetic Actually Human (2)
replace missing bolts                    (6/7)
e/l trials on lh rear door               (6/7)
--------------------- Common Predicted Synthetic Actually Synthetic (5)
need clean out steering pump hose        (6/7)
parts washer pump chip sensor alarm      (6/7)
replacing needed 4 eng pump bolt         (6/7)
extending needed 4 mast dust flap        (6/7)
leak fuel from tilt cylinder hose        (6/7)
--------------------- Common Predicted Human Actually Human (4)
both swing lube solenoid seal valve u/s  (6/7)
fit drop bar on upper deck               (6/7)
replace u/s jockey wheel                 (6/7)
coolant leak front of engine             (7/7)

Total commonly labelled: 13


# Ranking Test
- Naturalness
- Correctness

In [20]:
# Generate 25 samples from each for ranking test
synthetic_dirty = load_sentences(HUMANISE_DATAPATH)
dirty_100 = synthetic_sample(synthetic_dirty, 100, 8)
random.shuffle(dirty_100)
dirty_100 = pd.DataFrame(dirty_100, columns=['sentence'])
dirty_25 = random.sample(dirty_100['sentence'].tolist(), 25)
dirty_25 = pd.DataFrame(dirty_25, columns=['sentence'])
dirty_25['label'] = 's'

human_data = load_sentences(HUMAN_DATAPATH)
human_25 = random.sample(human_data, 25)
human_25 = pd.DataFrame(human_25, columns=['sentence'])
human_25['label'] = 'h'
s
dirty_25.to_csv('dirty_25.csv', index=False)
human_25.to_csv('human_25.csv', index=False)

# Combine and shuffle human and synthetic data
rank = pd.concat([dirty_25, human_25])
rank = rank.sample(frac=1).reset_index(drop=True)
rank.to_csv('rank_label.csv', index=False)  # labels
rank['naturalness'] = ''
rank['correctness'] = ''
rank.drop(columns=['label']).to_csv('rank.csv', index=False)

In [44]:
def calculate_ranking(target, file):
    data = pd.read_csv(file)
    labels = pd.read_csv(target).set_index('sentence')['label']
    results = {'Human': {'naturalness': [], 'correctness': []},
               'Synthetic': {'naturalness': [], 'correctness': []}}

    for i, row in data.iterrows():
        label = 'Synthetic' if labels[row['sentence']] == 's' else 'Human'
        results[label]['naturalness'].append(row['naturalness'])
        results[label]['correctness'].append(row['correctness'])

    summary = {key: [np.mean(results[key]['naturalness']),
                     np.mean(results[key]['correctness'])] for key in results}

    df = pd.DataFrame(summary, index=['Naturalness', 'Correctness'])
    print(df)

In [45]:
calculate_ranking('Rank/rank_label.csv', 'Rank/rank_mh.csv')

             Human  Synthetic
Naturalness   4.44       3.80
Correctness   4.24       4.04
